# Extract Frozen DINOv3 7B Features

Run this notebook once to extract a reusable feature archive from the competition images. The model is used only for inference; it is not trained. The archive stores all features required by the baseline and ablations: scales 256/512/768, the final layer and blocks 35/38, and average/max/min/signed-GeM pooled vectors.

Save a Kaggle Notebook version with its output, then attach that output dataset to the baseline and ablation notebooks. Training notebooks search /kaggle/input/*/dinov3_features.npz automatically. If running elsewhere, set CFG.FEATURE_FILE to the archive path.

## Kaggle inputs
Attach the competition data at /kaggle/input/csiro-biomass and the DINOv3 7B checkpoint at /kaggle/input/csiro-dinov3-patch16-7b/vit_7b_patch16_dinov3.lvd1689m. Enable a GPU before running this extraction notebook.


In [ ]:
# CSIRO Image2Biomass Prediction
# DINOv3 7B multiscale features with Ridge multiview stacking
# 1. Extract final and selected intermediate-layer features with average, max, min, and signed GeM pooling.
# 2. Apply RobustScaler and PCA independently to each feature view.
# 3. Train a Ridge base model per view, then stack view predictions with Ridge.

import os
import sys


import gc
import random
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T



from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.decomposition import PCA


# Suppress warnings
warnings.filterwarnings('ignore')
# ====================================================
# Configuration
# ====================================================
class CFG:
    DATA_PATH = '/kaggle/input/csiro-biomass'
    train_path = os.path.join(DATA_PATH, 'train.csv')
    test_path = os.path.join(DATA_PATH, 'test.csv')
    

    MODEL_DIR = '/kaggle/input/csiro-dinov3-patch16-7b/vit_7b_patch16_dinov3.lvd1689m'
    MODEL_NAME = f"vit_7b_patch16_dinov3.lvd1689m"
    
    # IMG_SIZES selects input scales; SPLIT_SCALES controls which scales use left-right crops.
    img_size = 512
    IMG_SIZES = [256, 512, 768]
    SPLIT_SCALES = (256,512,768)
    # Outer ensemble fold counts; this changes training runs only and does not repeat feature extraction.
    ENSEMBLE_FOLDS = [2,3,5]
    # Higher input resolutions use more GPU memory; reduce the batch size if needed.
    batch_size = 1
    MID_BLOCKS = [35, 38]  # Extract features from these additional intermediate blocks.


    USE_PCA = True
    PCA_N_COMPONENTS = 355          # PCA output dimensions per view.
    PCA_WHITEN = False              # Whitening is usually unnecessary.
    PCA_SVD_SOLVER = "randomized"   # Faster for small sample counts and wide feature vectors.
    
    
    seed = 42
    n_fold = 4
    
    target_cols = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    target_weights = [0.1, 0.1, 0.1, 0.2, 0.5]
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ====================================================
# Utility functions
# ====================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

seed_everything(CFG.seed)



def weighted_r2_metric(y_true, y_pred):
    """Competition metric: weighted R2."""
    weights = np.array(CFG.target_weights)
    if y_true.ndim == 1: y_true = y_true.reshape(-1, 1)
    if y_pred.ndim == 1: y_pred = y_pred.reshape(-1, 1)
    
    ss_res = np.sum(weights * np.sum((y_true - y_pred)**2, axis=0))
    global_mean = np.average(np.mean(y_true, axis=0), weights=weights)
    ss_tot = np.sum(weights * np.sum((y_true - global_mean)**2, axis=0))
    return 1 - (ss_res / (ss_tot + 1e-8))

def post_process_biomass(df_preds):
    """Reconcile biomass predictions with their physical relationships."""
    ordered_cols = ["Dry_Green_g", "Dry_Clover_g", "Dry_Dead_g", "GDM_g", "Dry_Total_g"]
    if not all(col in df_preds.columns for col in ordered_cols):
        return df_preds
    
    Y = df_preds[ordered_cols].values.T
    C = np.array([[1, 1, 0, -1, 0], [0, 0, 1, 1, -1]], dtype=float)
    
    try:
        inv_CCt = np.linalg.inv(C @ C.T)
    except np.linalg.LinAlgError:
        inv_CCt = np.linalg.pinv(C @ C.T)
    
    P = np.eye(5) - C.T @ inv_CCt @ C
    Y_reconciled = (P @ Y).T.clip(min=0)
    
    df_out = df_preds.copy()
    df_out[ordered_cols] = Y_reconciled
    
    # Enforce the target identities after projection.
    df_out['GDM_g'] = df_out['Dry_Green_g'] + df_out['Dry_Clover_g']
    df_out['Dry_Total_g'] = df_out['GDM_g'] + df_out['Dry_Dead_g']
    return df_out

# ====================================================
# Data loading and image preprocessing
# ====================================================
def load_data():
    train_df = pd.read_csv(CFG.train_path)
    test_df = pd.read_csv(CFG.test_path)
    
    def parse_ids(df):
        if 'sample_id' in df.columns:
            df[['image_id', 'target_type']] = df['sample_id'].str.split('__', n=1, expand=True)
        return df

    train_df = parse_ids(train_df)
    test_df = parse_ids(test_df)
    
    meta_cols = ['image_id', 'image_path']
    train_pivot = train_df.pivot_table(index=meta_cols, columns='target_name', values='target').reset_index()
    train_pivot.columns.name = None
    
    test_pivot = test_df[meta_cols].drop_duplicates().reset_index(drop=True)
    
    return train_pivot, test_pivot


def split_lr_pil(img: Image.Image, overlap: int = 0):
    """
    Split a PIL image into left and right crops with an optional pixel overlap.
    Returns the left and right image crops.
    """
    w, h = img.size
    mid = w // 2

    # Clamp crop boundaries to the image dimensions.
    l_right = min(w, mid + overlap)
    r_left  = max(0, mid - overlap)

    left  = img.crop((0, 0, l_right, h))
    right = img.crop((r_left, 0, w, h))
    return left, right


class BiomassDataset(Dataset):
    def __init__(self, df, transform=None, split_lr=False, overlap=0):
        self.image_paths = df['image_path'].values
        self.transform = transform
        self.split_lr = split_lr
        self.overlap = overlap
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        path = os.path.join(CFG.DATA_PATH, self.image_paths[idx])
        try:
            image = Image.open(path).convert("RGB")
        except:
            image = Image.new("RGB", (256, 256))

        if self.transform is None:
            return image

        if not self.split_lr:
            return self.transform(image)

        # Split left and right into two views.
        left, right = split_lr_pil(image, overlap=self.overlap)
        t_left  = self.transform(left)   # (3,H,W)
        t_right = self.transform(right)  # (3,H,W)
        return torch.stack([t_left, t_right], dim=0)  # (2,3,H,W)

# ====================================================
# DINOv3 checkpoint loading and feature extraction
# ====================================================
from pathlib import Path
import torch
import timm
from safetensors import safe_open

def load_timm_from_hf_dir_low_cpu(model_dir: str, model_name: str, device, dtype=torch.bfloat16):
    model_dir = Path(model_dir)

    # 0) Create the model on CPU and set its parameter dtype to bfloat16 before loading weights.
    model = timm.create_model(model_name, pretrained=False, num_classes=0, global_pool="")
    model = model.to(dtype=dtype)  # Convert model parameters to bfloat16 before loading the checkpoint.

    # 1) Load checkpoint weights on CPU.
    if (model_dir / "model.safetensors").exists():
        from safetensors.torch import load_file
        sd = load_file(str(model_dir / "model.safetensors"), device="cpu")
    else:
        sd = torch.load(str(model_dir / "pytorch_model.bin"), map_location="cpu")

    # 2) Load the weights using the model parameter dtype.
    incompat = model.load_state_dict(sd, strict=False)
    print("missing:", len(incompat.missing_keys), "unexpected:", len(incompat.unexpected_keys))

    # 3) Move the model to the target device.
    model = model.to(device).eval()
    return model
    

def debug_blocks_tokens(model, H=512, W=1024, block_indices=(35, 38)):
    p0 = next(model.parameters())
    x = torch.randn(1, 3, H, W, device=p0.device, dtype=p0.dtype)

    feats = {}
    handles = []
    for idx in block_indices:
        def _make_hook(i):
            def _hook(_m, _inp, out):
                feats[i] = out.detach()   # Detach the captured output so it remains a regular tensor.
            return _hook
        handles.append(model.blocks[idx].register_forward_hook(_make_hook(idx)))

    with torch.no_grad():                 # Use no_grad here because the captured block outputs are reused.
        out_last = model.forward_features(x)

    for h in handles:
        h.remove()

    print("forward_features(last) =", tuple(out_last.shape), out_last.dtype)
    for idx in block_indices:
        t = feats[idx]
        print(f"block[{idx}] output =", tuple(t.shape), t.dtype)

    if 39 in feats and hasattr(model, "norm") and model.norm is not None:
        diff = (out_last - model.norm(feats[39])).abs().mean().item()
        print("mean(|out_last - norm(block[39])|) =", diff)


def forward_with_blocks(model, imgs, block_indices, apply_final_norm_on_blocks=True):
    feats = {}
    handles = []

    # Remove duplicate block indices while preserving order.
    seen = set()
    block_indices = [b for b in block_indices if not (b in seen or seen.add(b))]

    for idx in block_indices:
        def _make_hook(i):
            def _hook(_m, _inp, out):
                # Handle implementations that return tuple or list outputs.
                if isinstance(out, (tuple, list)):
                    out = out[0]
                feats[i] = out.detach()
            return _hook
        handles.append(model.blocks[idx].register_forward_hook(_make_hook(idx)))

    with torch.no_grad():
        out_last = model.forward_features(imgs)

    for h in handles:
        h.remove()

    if apply_final_norm_on_blocks and hasattr(model, "norm") and model.norm is not None:
        for k in feats:
            feats[k] = model.norm(feats[k])

    return out_last, feats



def gem_pool_1d(x: torch.Tensor, p: float = 3.0, eps: float = 1e-6, dim: int = 1, signed: bool = True):
    """
    x has shape (B, N, C); pool across the token dimension.
    Returns a tensor with shape (B, C).
    """
    if signed:
        # Compute the signed power mean, then apply the signed p-th root.
        y = (torch.sign(x) * x.abs().clamp_min(eps).pow(p)).mean(dim=dim)  # (B, C)
        return torch.sign(y) * y.abs().pow(1.0 / p)                        # (B, C)
    else:
        y = x.clamp_min(eps).pow(p).mean(dim=dim)                          # (B, C)
        return y.pow(1.0 / p)                                              # (B, C)
                                



def extract_features_multiscale(df, model, mid_blocks, img_sizes=(256, 512),
                                split_scales=(768,), overlap=0):
    # Normalize mid_blocks to an ordered list of unique block indices.
    if isinstance(mid_blocks, int):
        mid_blocks = [mid_blocks]
    seen = set()
    mid_blocks = [b for b in list(mid_blocks) if not (b in seen or seen.add(b))]

    print(f"Extracting multi-scale features for {len(df)} images... scales={list(img_sizes)} mid_blocks={mid_blocks}")

    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    out = {}
    prefix = getattr(model, "num_prefix_tokens", 1)

    for sz in img_sizes:
        do_split = (sz in split_scales)

        print(f"\n[Scale {sz}] extracting last + mid blocks ... split_lr={do_split}")

        if do_split:
            # For split scales, crop the image first and resize each half to (sz, sz).
            transform = T.Compose([
                T.Resize((sz, sz)),
                T.ToTensor(),
                T.Normalize(mean, std),
            ])
            ds = BiomassDataset(df, transform=transform, split_lr=True, overlap=overlap)
        else:
            # For other scales, resize the full image to (sz, 2 * sz).
            transform = T.Compose([
                T.Resize((sz, sz * 2)),
                T.ToTensor(),
                T.Normalize(mean, std),
            ])
            ds = BiomassDataset(df, transform=transform, split_lr=False)

        ld = DataLoader(ds, batch_size=CFG.batch_size, shuffle=False, num_workers=0)

        last_epoch = []
        mid_epoch = {b: [] for b in mid_blocks}

        for batch in tqdm(ld, desc=f"Scale {sz}"):
            p = next(model.parameters())

            with torch.inference_mode():
                if batch.dim() == 5:
                    # Split mode batch shape: (B, 2, 3, sz, sz).
                    B, K, C, H, W = batch.shape

                    # Forward each crop separately to reduce peak GPU memory.
                    last_patches = []
                    mid_patches = {b: [] for b in mid_blocks}

                    for k in range(K):
                        imgs = batch[:, k].to(device=p.device, non_blocking=True)
                        if imgs.dtype != p.dtype:
                            imgs = imgs.to(dtype=p.dtype)

                        out_last_k, feats_k = forward_with_blocks(
                            model,
                            imgs,
                            block_indices=mid_blocks,
                            apply_final_norm_on_blocks=True
                        )

                        # Keep patch tokens only and remove prefix tokens.
                        last_patches.append(out_last_k[:, prefix:, :])  # (B, Nk, C)

                        for b in mid_blocks:
                            mid_patches[b].append(feats_k[b][:, prefix:, :])  # (B, Nk, C)

                    # Concatenate patch tokens from both crops along the token dimension.
                    last_patch = torch.cat(last_patches, dim=1)  # (B, N_total, C)

                    last_avg = last_patch.mean(dim=1)
                    last_max = last_patch.max(dim=1)[0]
                    last_min = last_patch.min(dim=1)[0]
                    last_gem = gem_pool_1d(last_patch, p=3.0, signed=True)

                    emb_last = torch.cat([last_avg, last_max, last_min, last_gem], dim=1)  # (B, 4C)

                    for b in mid_blocks:
                        mid_patch = torch.cat(mid_patches[b], dim=1)  # (B, N_total, C)
                        mid_avg = mid_patch.mean(dim=1)
                        mid_max = mid_patch.max(dim=1)[0]
                        mid_min = mid_patch.min(dim=1)[0]
                        mid_gem = gem_pool_1d(mid_patch, p=3.0, signed=True)

                        emb_mid = torch.cat([mid_avg, mid_max, mid_min, mid_gem], dim=1)  # (B, 4C)
                        mid_epoch[b].append(emb_mid.detach().float().cpu().numpy())

                else:
                    # Non-split batch shape: (B, 3, sz, 2 * sz).
                    imgs = batch.to(device=p.device, non_blocking=True)
                    if imgs.dtype != p.dtype:
                        imgs = imgs.to(dtype=p.dtype)

                    out_last, feats = forward_with_blocks(
                        model,
                        imgs,
                        block_indices=mid_blocks,
                        apply_final_norm_on_blocks=True
                    )

                    last_patch = out_last[:, prefix:, :]  # (B, N, C)
                    last_avg = last_patch.mean(dim=1)
                    last_max = last_patch.max(dim=1)[0]
                    last_min = last_patch.min(dim=1)[0]
                    last_gem = gem_pool_1d(last_patch, p=3.0, signed=True)

                    emb_last = torch.cat([last_avg, last_max, last_min, last_gem], dim=1)  # (B, 4C)

                    for b in mid_blocks:
                        tok = feats[b]
                        mid_patch = tok[:, prefix:, :]
                        mid_avg = mid_patch.mean(dim=1)
                        mid_max = mid_patch.max(dim=1)[0]
                        mid_min = mid_patch.min(dim=1)[0]
                        mid_gem = gem_pool_1d(mid_patch, p=3.0, signed=True)

                        emb_mid = torch.cat([mid_avg, mid_max, mid_min, mid_gem], dim=1)  # (B, 4C)
                        mid_epoch[b].append(emb_mid.detach().float().cpu().numpy())

            last_epoch.append(emb_last.detach().float().cpu().numpy())

        final_last = np.concatenate(last_epoch, axis=0)
        out[f"s{sz}_last"] = pd.DataFrame(
            final_last,
            columns=[f"s{sz}_last_feat_{i}" for i in range(final_last.shape[1])]
        )

        for b in mid_blocks:
            final_mid = np.concatenate(mid_epoch[b], axis=0)
            out[f"s{sz}_b{b}"] = pd.DataFrame(
                final_mid,
                columns=[f"s{sz}_b{b}_feat_{i}" for i in range(final_mid.shape[1])]
            )

    return out


def save_feature_archive(train_df, test_df, train_views, test_views, output_path):
    manifest = {
        "model_name": CFG.MODEL_NAME,
        "pool_order": ["avg", "max", "min", "gem_p3_signed"],
        "scales": list(CFG.IMG_SIZES),
        "split_scales": list(CFG.SPLIT_SCALES),
        "mid_blocks": list(CFG.MID_BLOCKS),
        "feature_layout": "Each view concatenates four pooled vectors in pool_order.",
        "targets": list(CFG.target_cols),
    }
    arrays = {
        "manifest_json": np.asarray(json.dumps(manifest)),
        "train_image_ids": train_df["image_id"].astype(str).to_numpy(),
        "test_image_ids": test_df["image_id"].astype(str).to_numpy(),
        "y_train": train_df[CFG.target_cols].to_numpy(dtype=np.float32),
    }
    for view, frame in train_views.items():
        arrays[f"train__{view}"] = frame.to_numpy(dtype=np.float32, copy=False)
    for view, frame in test_views.items():
        arrays[f"test__{view}"] = frame.to_numpy(dtype=np.float32, copy=False)
    np.savez_compressed(output_path, **arrays)
    print(f"Saved feature archive: {output_path} ({os.path.getsize(output_path) / 1024**3:.2f} GiB)")


def main():
    train_df, test_df = load_data()
    dtype = torch.bfloat16 if CFG.device.type == "cuda" else torch.float32
    model = load_timm_from_hf_dir_low_cpu(
        model_dir=CFG.MODEL_DIR, model_name=CFG.MODEL_NAME, device=CFG.device, dtype=dtype
    )
    train_views = extract_features_multiscale(
        train_df, model, mid_blocks=CFG.MID_BLOCKS, img_sizes=CFG.IMG_SIZES,
        split_scales=CFG.SPLIT_SCALES, overlap=0
    )
    test_views = extract_features_multiscale(
        test_df, model, mid_blocks=CFG.MID_BLOCKS, img_sizes=CFG.IMG_SIZES,
        split_scales=CFG.SPLIT_SCALES, overlap=0
    )
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
    output_dir.mkdir(parents=True, exist_ok=True)
    save_feature_archive(train_df, test_df, train_views, test_views, output_dir / "dinov3_features.npz")


if __name__ == "__main__":
    main()